In [96]:
from openai import OpenAI
from pathlib import Path
from IPython.display import Markdown, display, update_display
import sys
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import helpers.scrapers as hs

In [71]:
import importlib
importlib.reload(hs)

<module 'helpers.scrapers' from '/Users/milenakowalska/Desktop/AI/portfolio/ai_summarizer/helpers/scrapers.py'>

In [56]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="local")
MODEL = "llama3"

In [78]:
content_and_links = hs.fetch_website_content_and_links("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions")

In [79]:
content_and_links

{'title': 'Stanley Tucci explores Italy through its most timeless dishes | National Geographic',
 'content': "TRAVEL\nStanley Tucci explores Italy through its most timeless dishes\nThe second season of\nTucci in Italy\nfollows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.\nStanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series\nTucci in Italy\n.\nNational Geographic/Matt Holyoak\nBy\nElena Giardina\nLast updated May 12, 2026\nStanley Tucci\nis no stranger to the wonders and complexities of Italy.\nHe’s\ntraversed the country’s\nmost famous and obscure locales\n(\nFlorence\nand\nRome\n,\nMaremma and Senarica)\nand has\nwritten about its\nirresistible food\nin his cookbooks and memoirs. But\nthere are always new things to\ndiscover\nin\na place where culture and cuisine are inseparable.\nIn a second season of\nTucci in Italy\n, the\nactor returns to his\nancestral homeland to 

In [81]:
link_system_prompt = """
    You are provided with an article title and a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a summary of the given article,
    such as links to an author, or pages clarifying the history or the context of the topic.
    Please note that if you chose a relative link as relevant, you should build a full url out of it. 
    For example, if the chosen url is "/about-author" and the website you are analyzing is https://full.url/article-123, 
    you should build as a response https://full.url/about-author.
    You should respond in JSON as in this example (replace topic_name with the actual topic):

    {
        "links": [
            {"type": "author page", "url": "https://full.url/goes/here/about"},
            {"type": "context - topic (topic_name)", "url": "https://another.full.url/context"}
        ]
    }
"""

In [82]:
def get_links_user_prompt(content_and_links):
    user_prompt = f"""
You are preparing context in order to summarize the article with the title: {content_and_links["title"]}.
Here is the list of links on the website  {content_and_links["url"]} -
Please decide which of these are relevant web links for a summary of the article, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(content_and_links["links"])
    return user_prompt

In [83]:
def select_relevant_links(content_and_links):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(content_and_links)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [84]:
select_relevant_links(content_and_links)

{'links': [{'type': 'author page',
   'url': 'https://www.nationalgeographic.com/travel/article/stanley-tucci-interview-tucci-in-italy'},
  {'type': 'related topic - food culture',
   'url': 'https://www.nationalgeographic.com/related/d9881753-b9f1-37a5-81e3-90ce36e10a3c/food-culture'}]}

In [85]:
def fetch_page_and_all_relevant_links(content_and_links):
    relevant_links = select_relevant_links(content_and_links)
    result = f"##Article title: \n\n{content_and_links["title"]} \n##Landing Page:\n\n{content_and_links["content"]}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += hs.fetch_website_content(link["url"])
    return result

In [86]:
print(fetch_page_and_all_relevant_links(content_and_links))

##Article title: 

Stanley Tucci explores Italy through its most timeless dishes | National Geographic 
##Landing Page:

TRAVEL
Stanley Tucci explores Italy through its most timeless dishes
The second season of
Tucci in Italy
follows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.
Stanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series
Tucci in Italy
.
National Geographic/Matt Holyoak
By
Elena Giardina
Last updated May 12, 2026
Stanley Tucci
is no stranger to the wonders and complexities of Italy.
He’s
traversed the country’s
most famous and obscure locales
(
Florence
and
Rome
,
Maremma and Senarica)
and has
written about its
irresistible food
in his cookbooks and memoirs. But
there are always new things to
discover
in
a place where culture and cuisine are inseparable.
In a second season of
Tucci in Italy
, the
actor returns to his
ancestral homeland to explore Naples and Campa

In [87]:
tutorial_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages to a given main article/topic
and creates a short summary / learning tutorial about the topic and its context.
Respond in markdown without code blocks.
Include details of the history, article's author and wide context of the topic, but only if you have the information.
"""

In [92]:
def get_tutorial_user_prompt(url):
    content_and_links = hs.fetch_website_content_and_links(url)
    page_and_links = fetch_page_and_all_relevant_links(content_and_links)
    user_prompt = f"""
You are looking at a company called: {content_and_links["title"]}
Here are the contents of the article's landing page and other relevant pages;
use this information to build a short summary / learning tutorial about the main topics from the article.\n\n
"""
    user_prompt += page_and_links
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [93]:
get_tutorial_user_prompt("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions")

"\nYou are looking at a company called: Stanley Tucci explores Italy through its most timeless dishes | National Geographic\nHere are the contents of the article's landing page and other relevant pages;\nuse this information to build a short summary / learning tutorial about the main topics from the article.\n\n\n##Article title: \n\nStanley Tucci explores Italy through its most timeless dishes | National Geographic \n##Landing Page:\n\nTRAVEL\nStanley Tucci explores Italy through its most timeless dishes\nThe second season of\nTucci in Italy\nfollows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.\nStanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series\nTucci in Italy\n.\nNational Geographic/Matt Holyoak\nBy\nElena Giardina\nLast updated May 12, 2026\nStanley Tucci\nis no stranger to the wonders and complexities of Italy.\nHe’s\ntraversed the country’s\nmost famous and obscur

In [98]:
def create_tutorial(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": tutorial_system_prompt},
            {"role": "user", "content": get_tutorial_user_prompt(url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [99]:
create_tutorial("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions")

**Learning Tutorial: Italian Cuisine and Culture**

Italy is renowned for its rich culinary heritage, and Stanley Tucci's series "Tucci in Italy" takes us on a journey through the country's most timeless dishes. In this tutorial, we'll explore the history, traditions, and secrets behind some of Italy's iconic cuisine.

**History and Cultural Context**

As a second-generation Italian American, Stanley Tucci returns to his ancestral homeland to explore the culinary traditions of five distinct regions: Naples and Campania, Veneto, Le Marche, Sicily, and Sardinia. Each region has its unique history, geography, and cultural influences that shape its cuisine.

**The Power of Food**

Tucci believes that food has the power to bring people together, transcending politics, ideals, and religions. Italian cuisine is no exception, with dishes often reflecting local traditions, family recipes, and cultural heritage. From simple ingredients to decadent feasts, every meal tells a story about Italy's past and present.

**Geographical Features**

Sardinia, in particular, leaves a deep impression on Tucci, with its stunning natural beauty and unique gastronomic traditions. The region's interior is home to ancient history, pristine landscapes, and a people who have maintained their traditional ways of life and foodways.

**Regional Traditions**

Throughout the series, Tucci explores the distinct culinary traditions of each region. In Campania, he discovers innovative dishes like spaghettino alle vongole fujute, which was historically made with stones from the sea to give it a salty flavor. He also meets people who share stories about their families' recipes and cultural practices.

**Authentic Flavor**

The secret to Italy's heavenly blood orange salad? Freshly squeezed juice! Tucci shares his love for the country's timeless dishes, emphasizing the importance of preserving traditions while also exploring new flavors and techniques.

**Conclusion**

"Tucci in Italy" is a love letter to food, family, tradition, and culture. By exploring Italian cuisine, we gain insight into the history, people, and landscapes that shape this country's gastronomic identity. In an increasingly complex world, Tucci shows us that food can bring us together, fostering connections and understanding across cultures and borders.

**Key Takeaways**

* Italian cuisine is rooted in local traditions, family recipes, and cultural heritage.
* Every meal tells a story about Italy's past and present.
* Food has the power to bring people together, transcending differences and divisions.
* Sardinia is a highlight of the series, with its unique gastronomic traditions and stunning natural beauty.